```
# =============================================================================
# Integrantes: Arthur Meneses Neves, Danilo Oliveira Santos, Matheus Gabriel
# Viana Araujo, João Victor Vidal Barbosa, Guilherme Araujo Castro
#
# Síntese: Resultados finais da N2. Ajuste de hiperparâmetros de Random Forest
# e XGBoost com o mesmo orçamento, variação por semente, estudo de ablação
# dos grupos de atributos (sociodemográfico, tempo, estado civil,
# circunstância do óbito), modelos finais nos conjuntos base e ampliado,
# intervalos de confiança por bootstrap pareado, baseline por tabela
# (idade x sexo), análise de erro por subgrupo com supressão de células
# pequenas, importância por permutação (XAI) e checagem temporal.
# A lógica reutilizável está em src/sim_utils.py e src/avaliacao.py.
#
# Changelog:
#   2026-11-16 - Matheus Araujo - criação do notebook (protocolo da N2)
# =============================================================================
```

# Resultados finais (N2): ajuste, atributos ampliados, incerteza e equidade

**O que muda em relação à N1 (notebook 03).** A N1 terminou com F1-macro de 0,1406 (XGBoost) e 0,1329 (Random Forest), contra 0,0224 do baseline, e deixou seis limitações explícitas. Este notebook ataca cada uma:

| Limitação da N1 | O que este notebook faz |
|---|---|
| (i) uma divisão e uma semente, sem intervalo de confiança | bootstrap pareado no teste (IC 95%) e repetição do treino com 3 sementes |
| (ii) hiperparâmetros não ajustados, profundidades diferentes | busca aleatória com o **mesmo orçamento** para RF e XGBoost, em validação separada do teste |
| (iii) capítulos VII e VIII com poucos casos | amostras com piso por classe; resultados por classe com o suporte ao lado |
| (iv) sem análise de erro por subgrupo | métricas por sexo, raça/cor, faixa etária, escolaridade e região |
| (v) `ESC = 0` mapeado para "Ignorado" | contagem de `ESC = 0` por ano para fundamentar a decisão |
| (vi) supressão de células < 10 não implementada | `avaliacao.suprimir_celulas_pequenas`, aplicada a todas as tabelas por subgrupo |

A N1 concluiu que "o limite observado parece vir do conjunto de atributos, não da escolha do algoritmo". Para testar essa hipótese, a N2 compara **dois conjuntos de entrada**: o **base** (as mesmas 5 variáveis da N1) e o **ampliado** (base + mês e ano do óbito + estado civil + local de ocorrência + assistência médica), e mede a contribuição de cada grupo por ablação.

**Protocolo (fixado antes de rodar):**

1. **Dados e divisão:** mesma preparação (`sim_utils.carregar_df_modelo`) e **mesma divisão 80/20 estratificada da N1** (`random_state=42`), logo o mesmo conjunto de teste. O teste só é usado na seção 7 em diante.
2. **Ajuste:** busca aleatória com `N_ITER_BUSCA` configurações por modelo, treinadas numa amostra estratificada do treino e avaliadas numa **validação** separada, também tirada do treino. Critério: F1-macro de validação.
3. **Desbalanceamento:** pesos de amostra balanceados, como na N1.
4. **Modelos finais:** RF e XGBoost com os melhores parâmetros, nos conjuntos base e ampliado, treinados numa fração `FRACAO_TREINO` do treino (amostra estratificada) e avaliados no teste completo.
5. **Métricas:** F1-macro (critério principal) com IC 95% por bootstrap pareado; recall, precisão e F1 por classe; matriz de confusão; acurácia só como referência.
6. **Baselines:** classe majoritária (o da N1) e **tabela idade × sexo** (prevê, para cada faixa de 5 anos e sexo, a classe de maior frequência ponderada). Um modelo de IA só se justifica se superar a tabela.
7. **Equidade e explicabilidade:** métricas por subgrupo com supressão de células com menos de 10 casos; importância por permutação dos atributos originais.
8. **Checagem temporal:** treino ≤ 2022, teste 2023 a 2026, como na N1.

In [ ]:
import json
import os
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

sys.path.insert(0, str(Path.cwd().parent / "src"))
import avaliacao as av
from sim_utils import (
    FEATURE_COLUMNS,
    FEATURE_COLUMNS_AMPLIADO,
    GRUPOS_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    carregar_df_modelo,
    regiao_de_uf,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)


def cfg(nome, padrao):
    """Lê a configuração de uma variável de ambiente, se existir (permite
    rodar o notebook inteiro com dados sintéticos para teste)."""
    bruto = os.environ.get(nome)
    return padrao if bruto is None else type(padrao)(bruto)


# ---- Configuração (ajuste aqui conforme a máquina) ---------------------------
DATA_DIR = Path(cfg("SIM_DATA_DIR", "../data"))
CACHE_DIR = DATA_DIR / "cache"
RESULTADOS_DIR = Path(cfg("SIM_RESULTADOS_DIR", "../resultados"))
FIG_DIR = Path(cfg("SIM_FIG_DIR", "../docs/figuras"))
ANOS = list(range(2000, 2027))

SEED = 42
FRACAO_TREINO = cfg("FRACAO_TREINO", 0.3)      # fração do treino nos modelos finais (1.0 = tudo)
N_BUSCA_TREINO = cfg("N_BUSCA_TREINO", 1_000_000)  # linhas de treino na busca
N_BUSCA_VAL = cfg("N_BUSCA_VAL", 500_000)          # linhas de validação na busca
N_ITER_BUSCA = cfg("N_ITER_BUSCA", 10)             # configurações por modelo (mesmo orçamento)
SEMENTES = [42, 7, 2026]
N_BOOT = cfg("N_BOOT", 200)
N_PERMUTACAO = cfg("N_PERMUTACAO", 300_000)        # linhas do teste na importância por permutação
N_REP_PERMUTACAO = cfg("N_REP_PERMUTACAO", 3)
RODAR_TEMPORAL = cfg("RODAR_TEMPORAL", 1) == 1
N_JOBS = -1
MODO_RAPIDO = cfg("SIM_MODO_RAPIDO", 0) == 1       # só para o teste com dados sintéticos

RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

NOMES_CURTOS = {
    "I": "infecciosas", "II": "neoplasias", "III": "sangue/imunit.", "IV": "endócrinas",
    "V": "mentais", "VI": "nervoso", "VII": "olho", "VIII": "ouvido",
    "IX": "circulatório", "X": "respiratório", "XI": "digestivo", "XII": "pele",
    "XIII": "osteomuscular", "XIV": "geniturinário", "XV": "gravidez/parto",
    "XVI": "perinatal", "XVII": "malformações", "XVIII": "mal definidas",
    "XX": "causas externas",
}
# Resultados da N1 (notebook 03), só para comparação
N1 = {"Baseline majoritária (N1)": 0.0224, "Random Forest (N1)": 0.1329, "XGBoost (N1)": 0.1406}

try:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
except (AttributeError, ValueError, OSError):
    ram_gb = float("nan")  # Windows: conferir no Gerenciador de Tarefas
print(f"Python {platform.python_version()} | scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | pandas {pd.__version__}")
print(f"CPU: {os.cpu_count()} núcleos | RAM: {ram_gb:.1f} GB | FRACAO_TREINO = {FRACAO_TREINO} | MODO_RAPIDO = {MODO_RAPIDO}")

## 1. Dados: novas colunas, valores "Ignorado" por ano e o caso `ESC = 0`

O conjunto ampliado usa três colunas opcionais do SIM (`ESTCIV`, `LOCOCOR`, `ASSISTMED`) e o mês de `DTOBITO`. Todas são do bloco de identificação e ocorrência da Declaração de Óbito, preenchidas independentemente da codificação da causa. `NECROPSIA` fica de fora: a necropsia é indicada justamente quando se suspeita de morte violenta, então ela é consequência da causa (vazamento indireto). Se alguma coluna faltar em algum ano, ela vira "Ignorado" nesse ano; a célula abaixo mostra quais anos faltam e o percentual de "Ignorado" por ano.

In [ ]:
t0 = time.perf_counter()
df = carregar_df_modelo(DATA_DIR, ANOS, CACHE_DIR)
diag = df.attrs["diagnosticos"]
print(f"Carga: {time.perf_counter() - t0:.1f} s | {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print("Contabilidade da preparação:", diag["contabilidade"])
print("Colunas opcionais ausentes por ano:", diag.get("opcionais_ausentes") or "nenhuma")

ignorado_por_ano = (
    df.groupby("ano_arquivo", observed=True)[["racacor", "escolaridade", "estciv", "lococor", "assistmed", "mes_obito"]]
    .agg(lambda s: 100 * (s.astype(str) == "Ignorado").mean())
    .round(1)
)
print()
print('% de "Ignorado" por ano:')
print(ignorado_por_ano.to_string())
ignorado_por_ano.to_csv(RESULTADOS_DIR / "n2_ignorado_por_ano.csv")

In [ ]:
# ESC = 0 não consta do dicionário da variável ESC (1 a 5 e 9). A partir de
# 2010 o SIM também tem ESC2010, em que 0 significa "sem escolaridade". Se
# ESC = 0 aparecer só em alguns anos e em volume grande, ele pode estar sendo
# usado como "nenhuma"; se for raro, mapear para "Ignorado" tem efeito
# desprezível. A contagem abaixo decide qual dos casos vale.
esc = pd.DataFrame(diag["esc_bruto_por_ano"]).T.fillna(0).astype(int)
esc.index = esc.index.astype(int)
esc = esc.sort_index()
esc_zero = pd.DataFrame({
    "esc_0": esc.get("0", pd.Series(0, index=esc.index)),
    "total": esc.sum(axis=1),
})
esc_zero["pct_esc_0"] = (100 * esc_zero["esc_0"] / esc_zero["total"]).round(3)
print(esc_zero.to_string())
print(f"\nTotal com ESC = 0: {esc_zero['esc_0'].sum():,} ({100 * esc_zero['esc_0'].sum() / esc_zero['total'].sum():.3f}% do bruto)")
esc_zero.to_csv(RESULTADOS_DIR / "n2_esc_zero_por_ano.csv")

## 2. Divisão treino/teste (a mesma da N1) e amostras de busca e de treino final

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df[TARGET_COLUMN].astype(str))
k = len(le.classes_)
rotulos = [f"{c} {NOMES_CURTOS.get(c, '')}" for c in le.classes_]

# Mesma chamada do notebook 03: mesmo conjunto de teste.
idx_treino, idx_teste = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED, stratify=y)
y_teste = y[idx_teste]

# Busca de hiperparâmetros: treino e validação disjuntos, ambos dentro do treino.
y_treino = y[idx_treino]
pos_busca_tr = av.amostra_estratificada(y_treino, N_BUSCA_TREINO, seed=SEED)
pos_busca_val = av.amostra_estratificada(y_treino, N_BUSCA_VAL, seed=SEED + 1, excluir=pos_busca_tr)
idx_busca_tr, idx_busca_val = idx_treino[pos_busca_tr], idx_treino[pos_busca_val]

# Treino dos modelos finais
if FRACAO_TREINO >= 1:
    idx_final = idx_treino
else:
    idx_final = idx_treino[av.amostra_estratificada(y_treino, int(FRACAO_TREINO * len(idx_treino)), seed=SEED + 2)]

assert np.intersect1d(idx_busca_tr, idx_busca_val).size == 0
assert np.intersect1d(np.concatenate([idx_busca_tr, idx_busca_val, idx_final]), idx_teste).size == 0
print(f"Classes: {k} | treino: {len(idx_treino):,} | teste: {len(idx_teste):,}")
print(f"Busca: treino {len(idx_busca_tr):,} + validação {len(idx_busca_val):,} | treino final: {len(idx_final):,} ({FRACAO_TREINO:.0%} do treino)")
print("Menor classe na busca (treino/val):", np.bincount(y[idx_busca_tr], minlength=k).min(), "/", np.bincount(y[idx_busca_val], minlength=k).min())

## 3. Pré-processamento

Mesmo esquema da N1: *one-hot* denso em `float32` para as categóricas e números passando direto. As categorias do *one-hot* vêm do tipo `category` do pandas (lista fixa definida na preparação), então o codificador é o mesmo em qualquer amostra e não depende do teste.

In [ ]:
def montar_preprocessador(colunas):
    categoricas = [c for c in colunas if c not in NUMERIC_FEATURES]
    numericas = [c for c in colunas if c in NUMERIC_FEATURES]
    categorias = [list(df[c].cat.categories) for c in categoricas]
    return ColumnTransformer(
        [
            ("cat", OneHotEncoder(categories=categorias, handle_unknown="ignore", sparse_output=False, dtype=np.float32), categoricas),
            ("num", "passthrough", numericas),
        ],
        sparse_threshold=0,
    ).fit(df.iloc[:1000][colunas])


def matriz(prep, idx, colunas):
    return prep.transform(df.iloc[idx][colunas]).astype(np.float32, copy=False)


CONJUNTOS = {"base": FEATURE_COLUMNS, "ampliado": FEATURE_COLUMNS_AMPLIADO}
PREP = {nome: montar_preprocessador(cols) for nome, cols in CONJUNTOS.items()}
for nome, cols in CONJUNTOS.items():
    n_col = len(PREP[nome].get_feature_names_out())
    gb = len(idx_final) * n_col * 4 / 1e9
    print(f"{nome:9s}: {len(cols)} atributos -> {n_col} colunas | matriz de treino final ~{gb:.1f} GB, teste ~{len(idx_teste) * n_col * 4 / 1e9:.1f} GB")

## 4. Busca de hiperparâmetros (mesmo orçamento para os dois modelos)

Os espaços de busca ficam fixos antes de rodar. No Random Forest, a profundidade máxima vai até 16 e as folhas têm no mínimo 50 linhas: com 19 classes, cada nó guarda 19 valores e árvores sem limite ocupariam dezenas de GB no treino final. O XGBoost usa `tree_method="hist"` como na N1. A busca usa o conjunto **ampliado**; os mesmos parâmetros são usados no conjunto base, para que a comparação base × ampliado mude só as entradas.

In [ ]:
RF_ESPACO = {
    "n_estimators": [100, 200],
    "max_depth": [10, 13, 16],
    "min_samples_leaf": [50, 200, 1000],
    "max_features": ["sqrt", 0.5],
}
XGB_ESPACO = {
    "n_estimators": [100, 200, 400],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.05, 0.1, 0.3],
    "min_child_weight": [1, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 1.0],
}
if MODO_RAPIDO:  # só para o teste com dados sintéticos
    RF_ESPACO = {"n_estimators": [10, 20], "max_depth": [4, 8], "min_samples_leaf": [5, 20], "max_features": ["sqrt"]}
    XGB_ESPACO = {"n_estimators": [10, 20], "max_depth": [3, 5], "learning_rate": [0.3], "min_child_weight": [1],
                  "subsample": [1.0], "colsample_bytree": [1.0]}


def construir_rf(seed=SEED, **p):
    return RandomForestClassifier(random_state=seed, n_jobs=N_JOBS, **p)


def construir_xgb(seed=SEED, **p):
    return XGBClassifier(objective="multi:softmax", num_class=k, tree_method="hist", random_state=seed, n_jobs=N_JOBS, **p)


CONSTRUTORES = {"Random Forest": construir_rf, "XGBoost": construir_xgb}
ESPACOS = {"Random Forest": RF_ESPACO, "XGBoost": XGB_ESPACO}

prep = PREP["ampliado"]
Xb_tr = matriz(prep, idx_busca_tr, FEATURE_COLUMNS_AMPLIADO)
Xb_val = matriz(prep, idx_busca_val, FEATURE_COLUMNS_AMPLIADO)
yb_tr, yb_val = y[idx_busca_tr], y[idx_busca_val]
wb_tr = compute_sample_weight("balanced", yb_tr)

BUSCA, MELHORES = {}, {}
for nome, construir in CONSTRUTORES.items():
    print(f"=== {nome} ===")
    t0 = time.perf_counter()
    BUSCA[nome] = av.busca_aleatoria(construir, ESPACOS[nome], N_ITER_BUSCA, SEED, Xb_tr, yb_tr, wb_tr, Xb_val, yb_val, k)
    MELHORES[nome] = av.melhores_parametros(BUSCA[nome], ESPACOS[nome])
    BUSCA[nome].to_csv(RESULTADOS_DIR / f"n2_busca_{'rf' if nome == 'Random Forest' else 'xgb'}.csv", index=False)
    print(f"Busca de {nome}: {time.perf_counter() - t0:.0f} s | melhor: {MELHORES[nome]} (F1-macro val = {BUSCA[nome]['f1_macro_val'].iloc[0]:.4f})\n")

In [ ]:
for nome in BUSCA:
    print(f"{nome}: 5 melhores configurações")
    print(BUSCA[nome].head(5).round(4).to_string(index=False))
    print()

## 5. Variação por semente

Uma única semente não diz se a diferença entre RF e XGBoost é maior que o ruído do próprio treino. Aqui os melhores modelos são retreinados com 3 sementes; em cada uma muda a semente do modelo **e** a amostra de treino da busca (a validação fica fixa).

In [ ]:
linhas = []
for semente in SEMENTES:
    pos = av.amostra_estratificada(y_treino, N_BUSCA_TREINO, seed=semente, excluir=pos_busca_val)
    X_s = matriz(prep, idx_treino[pos], FEATURE_COLUMNS_AMPLIADO)
    y_s = y_treino[pos]
    w_s = compute_sample_weight("balanced", y_s)
    for nome, construir in CONSTRUTORES.items():
        modelo = construir(seed=semente, **MELHORES[nome]).fit(X_s, y_s, sample_weight=w_s)
        linhas.append({"modelo": nome, "semente": semente, "f1_macro_val": av.f1_macro(yb_val, modelo.predict(Xb_val), k)})
    del X_s
sementes = pd.DataFrame(linhas)
resumo_sementes = sementes.groupby("modelo")["f1_macro_val"].agg(["mean", "std", "min", "max"])
print(sementes.pivot(index="semente", columns="modelo", values="f1_macro_val").round(4).to_string())
print()
print(resumo_sementes.round(4).to_string())
sementes.to_csv(RESULTADOS_DIR / "n2_sementes.csv", index=False)

## 6. Ablação: quanto cada grupo de atributos acrescenta

XGBoost com os melhores parâmetros, mesma amostra de busca e mesma validação. Os grupos entram de forma cumulativa, e também se mede o efeito de **retirar** cada grupo do conjunto ampliado. Se o ganho vier quase todo de "circunstância" (local de ocorrência e assistência), isso precisa ser discutido: local "via pública", por exemplo, está associado a causas externas, mas é uma informação observada no momento do óbito, não derivada da causa codificada.

In [ ]:
def f1_val_xgb(colunas):
    p = montar_preprocessador(colunas)
    modelo = construir_xgb(**MELHORES["XGBoost"]).fit(matriz(p, idx_busca_tr, colunas), yb_tr, sample_weight=wb_tr)
    return av.f1_macro(yb_val, modelo.predict(matriz(p, idx_busca_val, colunas)), k)


ablacao = []
acumulado = []
for grupo, cols in GRUPOS_FEATURES.items():
    acumulado = acumulado + cols
    ablacao.append({"experimento": "+ " + grupo if grupo != "sociodemografico" else "base (sociodemográfico)",
                    "tipo": "cumulativo", "n_atributos": len(acumulado), "f1_macro_val": f1_val_xgb(acumulado)})
completo = ablacao[-1]["f1_macro_val"]
for grupo in list(GRUPOS_FEATURES)[1:]:
    cols = [c for c in FEATURE_COLUMNS_AMPLIADO if c not in GRUPOS_FEATURES[grupo]]
    f1 = f1_val_xgb(cols)
    ablacao.append({"experimento": f"ampliado sem {grupo}", "tipo": "retirada", "n_atributos": len(cols),
                    "f1_macro_val": f1})
ablacao = pd.DataFrame(ablacao)
ablacao["ganho_vs_anterior"] = np.where(ablacao["tipo"] == "cumulativo", ablacao["f1_macro_val"].diff(), np.nan)
ablacao["perda_vs_completo"] = np.where(ablacao["tipo"] == "retirada", completo - ablacao["f1_macro_val"], np.nan)
print(ablacao.round(4).to_string(index=False))
ablacao.to_csv(RESULTADOS_DIR / "n2_ablacao.csv", index=False)
del Xb_tr

## 7. Baselines e modelos finais no conjunto de teste

Dois baselines e quatro modelos (RF e XGBoost × base e ampliado). Daqui em diante o conjunto de teste é usado pela primeira vez.

In [ ]:
PRED, TEMPO = {}, {}

# Baseline 1: classe majoritária (o mesmo da N1)
classe_maj = np.bincount(y[idx_final]).argmax()
PRED["Baseline majoritária"] = np.full(len(idx_teste), classe_maj)
TEMPO["Baseline majoritária"] = 0.0

# Baseline 2: tabela faixa de 5 anos x sexo, com pesos balanceados
t0 = time.perf_counter()
tabela = av.BaselineTabela().fit(df.iloc[idx_final][["idade_anos", "sexo"]], y[idx_final])
TEMPO["Baseline tabela idade×sexo"] = time.perf_counter() - t0
PRED["Baseline tabela idade×sexo"] = tabela.predict(df.iloc[idx_teste][["idade_anos", "sexo"]])
print(f"Tabela idade×sexo: {len(tabela.regra_)} células")

MODELOS = {}
pesos_final = compute_sample_weight("balanced", y[idx_final])
for conjunto, colunas in CONJUNTOS.items():
    X_tr = matriz(PREP[conjunto], idx_final, colunas)
    X_te = matriz(PREP[conjunto], idx_teste, colunas)
    for nome, construir in CONSTRUTORES.items():
        rotulo = f"{nome} ({conjunto})"
        t0 = time.perf_counter()
        modelo = construir(**MELHORES[nome]).fit(X_tr, y[idx_final], sample_weight=pesos_final)
        TEMPO[rotulo] = time.perf_counter() - t0
        PRED[rotulo] = modelo.predict(X_te).astype(np.int16)
        MODELOS[rotulo] = modelo
        print(f"{rotulo}: treino {TEMPO[rotulo]:.0f} s | F1-macro teste = {av.f1_macro(y_teste, PRED[rotulo], k):.4f}")
    del X_tr, X_te

In [ ]:
linhas = []
for nome, pred in PRED.items():
    m = av.metricas_de_confusao(av.matriz_confusao(y_teste, pred, k))
    linhas.append({"modelo": nome, "f1_macro": m["f1_macro"], "acuracia": m["acuracia"],
                   "recall_macro": m["recall_macro"], "tempo_treino_s": TEMPO[nome]})
comparacao = pd.DataFrame(linhas).set_index("modelo")

t0 = time.perf_counter()
boot = av.bootstrap_f1_macro(y_teste, PRED, k, n_boot=N_BOOT, seed=SEED)
ic = av.resumo_ic(boot)
comparacao["ic95_inf"] = ic["ic_inf"]
comparacao["ic95_sup"] = ic["ic_sup"]
print(f"Bootstrap ({N_BOOT} réplicas): {time.perf_counter() - t0:.0f} s\n")
print(comparacao.round(4).to_string())
print("\nReferência N1 (F1-macro):", N1)
comparacao.to_csv(RESULTADOS_DIR / "n2_comparacao.csv")

MELHOR = comparacao.drop(index=[n for n in comparacao.index if n.startswith("Baseline")])["f1_macro"].idxmax()
print(f"\nMelhor modelo no teste: {MELHOR}")

In [ ]:
pares = [
    ("XGBoost (ampliado)", "Random Forest (ampliado)"),
    ("XGBoost (base)", "Random Forest (base)"),
    ("XGBoost (ampliado)", "XGBoost (base)"),
    ("Random Forest (ampliado)", "Random Forest (base)"),
    (MELHOR, "Baseline tabela idade×sexo"),
]
dif = av.diferencas_pareadas(boot, pares)
resumo_dif = av.resumo_ic(dif)
resumo_dif["observado"] = [comparacao.loc[a, "f1_macro"] - comparacao.loc[b, "f1_macro"] for a, b in pares]
resumo_dif["ic_exclui_zero"] = (resumo_dif["ic_inf"] > 0) | (resumo_dif["ic_sup"] < 0)
print("Diferenças de F1-macro (bootstrap pareado, IC 95%):")
print(resumo_dif.round(4).to_string())
resumo_dif.to_csv(RESULTADOS_DIR / "n2_diferencas.csv")

## 8. Desempenho por classe

In [ ]:
suporte = np.bincount(y_teste, minlength=k)
por_classe = {"descrição": [NOMES_CURTOS.get(c, "") for c in le.classes_], "suporte_teste": suporte}
for nome in ["Baseline tabela idade×sexo", "Random Forest (base)", "XGBoost (base)", "Random Forest (ampliado)", "XGBoost (ampliado)"]:
    m = av.metricas_de_confusao(av.matriz_confusao(y_teste, PRED[nome], k))
    curto = nome.replace("Random Forest", "RF").replace("XGBoost", "XGB").replace("Baseline tabela idade×sexo", "tabela")
    por_classe[f"F1 {curto}"] = m["f1_por_classe"]
    por_classe[f"rec {curto}"] = m["recall_por_classe"]
por_classe = pd.DataFrame(por_classe, index=le.classes_).sort_values("suporte_teste", ascending=False)
por_classe.index.name = "capítulo"
print(por_classe.round(3).to_string())
por_classe.to_csv(RESULTADOS_DIR / "n2_por_classe.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 11))
ConfusionMatrixDisplay.from_predictions(
    y_teste, PRED[MELHOR], labels=range(k), display_labels=rotulos, normalize="true",
    values_format=".2f", cmap="Blues", ax=ax, xticks_rotation=90, text_kw={"fontsize": 6},
)
ax.set_title(f"{MELHOR}: matriz de confusão normalizada por linha")
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe verdadeira")
fig.tight_layout()
fig.savefig(FIG_DIR / "n2_matriz_confusao.png", dpi=200)
plt.show()

In [ ]:
ordem = por_classe.index
fig, ax = plt.subplots(figsize=(11, 5))
largura = 0.27
x = np.arange(len(ordem))
for i, (col, nome) in enumerate([("F1 tabela", "Tabela idade×sexo"), ("F1 XGB (base)", "XGBoost base"), ("F1 XGB (ampliado)", "XGBoost ampliado")]):
    ax.bar(x + (i - 1) * largura, por_classe.loc[ordem, col], largura, label=nome)
ax.set_xticks(x, [f"{c}\n{NOMES_CURTOS.get(c, '')}" for c in ordem], rotation=90, fontsize=8)
ax.set_ylabel("F1 no teste")
ax.set_title("F1 por capítulo (ordenado pelo suporte)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "n2_f1_por_classe.png", dpi=200)
plt.show()

## 9. Análise de erro por subgrupo (equidade)

Métricas do melhor modelo no teste, por sexo, raça/cor, faixa etária, escolaridade e região. O F1-macro de cada subgrupo considera só as classes que aparecem nele (gravidez não aparece entre homens, por exemplo). Células com menos de 10 casos têm métricas e contagem suprimidas.

Diferenças entre subgrupos aqui **não** são, sozinhas, prova de viés do modelo: os subgrupos têm composições de causas diferentes (a idade média de óbito difere entre grupos de raça/cor, por exemplo), e isso muda a dificuldade da tarefa. Por isso a tabela mostra também o número de classes presentes e a comparação com a tabela idade×sexo no mesmo subgrupo.

In [ ]:
teste_df = df.iloc[idx_teste]
SUBGRUPOS = {
    "sexo": teste_df["sexo"].astype(str).to_numpy(),
    "raça/cor": teste_df["racacor"].astype(str).to_numpy(),
    "faixa etária": av.faixa_etaria(teste_df["idade_anos"]),
    "escolaridade": teste_df["escolaridade"].astype(str).to_numpy(),
    "região": regiao_de_uf(teste_df["uf"].astype(str)).to_numpy(),
}
tabelas = []
for variavel, grupo in SUBGRUPOS.items():
    t_m = av.metricas_por_subgrupo(y_teste, PRED[MELHOR], grupo, k)
    t_b = av.metricas_por_subgrupo(y_teste, PRED["Baseline tabela idade×sexo"], grupo, k)
    t = t_m[["n", "classes_presentes", "f1_macro", "recall_macro", "acuracia", "suprimido"]].copy()
    t["f1_macro_tabela"] = t_b["f1_macro"]
    t["ganho_sobre_tabela"] = t["f1_macro"] - t["f1_macro_tabela"]
    t.insert(0, "variavel", variavel)
    tabelas.append(t)
subgrupos = pd.concat(tabelas)
subgrupos.index.name = "grupo"
print(subgrupos.round(4).to_string())
subgrupos.to_csv(RESULTADOS_DIR / "n2_subgrupos.csv")

amplitude = subgrupos.groupby("variavel")["f1_macro"].agg(lambda s: s.max() - s.min())
print("\nAmplitude do F1-macro entre grupos (máx - mín):")
print(amplitude.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, len(SUBGRUPOS), figsize=(16, 4), sharey=True)
for ax, (variavel, t) in zip(axes, subgrupos.groupby("variavel", sort=False)):
    t = t.dropna(subset=["f1_macro"])
    ax.bar(range(len(t)), t["f1_macro"], label="melhor modelo")
    ax.scatter(range(len(t)), t["f1_macro_tabela"], color="black", marker="_", s=200, label="tabela idade×sexo")
    ax.set_xticks(range(len(t)), t.index, rotation=90, fontsize=8)
    ax.set_title(variavel)
axes[0].set_ylabel("F1-macro no subgrupo")
axes[0].legend(fontsize=7)
fig.suptitle(f"{MELHOR}: F1-macro por subgrupo (células < 10 suprimidas)")
fig.tight_layout()
fig.savefig(FIG_DIR / "n2_subgrupos.png", dpi=200)
plt.show()

## 10. Explicabilidade: importância por permutação

Para cada atributo original (antes do *one-hot*), embaralha-se a coluna numa amostra do teste e mede-se a queda do F1-macro. Isso responde quais atributos o modelo realmente usa. Embaralhar a coluna original, e não cada coluna do *one-hot*, mede o atributo inteiro.

In [ ]:
modelo_melhor = MODELOS[MELHOR]
conjunto_melhor = "ampliado" if "ampliado" in MELHOR else "base"
cols_melhor = CONJUNTOS[conjunto_melhor]
amostra_perm = np.random.default_rng(SEED).choice(len(idx_teste), size=min(N_PERMUTACAO, len(idx_teste)), replace=False)
X_perm = df.iloc[idx_teste[amostra_perm]][cols_melhor].reset_index(drop=True)


def prever(X_df):
    return modelo_melhor.predict(PREP[conjunto_melhor].transform(X_df).astype(np.float32))


t0 = time.perf_counter()
importancia = av.importancia_por_permutacao(prever, X_perm, y_teste[amostra_perm], k, n_rep=N_REP_PERMUTACAO, seed=SEED)
print(f"{time.perf_counter() - t0:.0f} s | F1-macro sem permutação na amostra: {importancia.attrs['f1_base']:.4f}")
print(importancia.round(4).to_string(index=False))
importancia.to_csv(RESULTADOS_DIR / "n2_importancia.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
imp = importancia.iloc[::-1]
ax.barh(imp["atributo"], imp["queda_f1_media"], xerr=imp["queda_f1_dp"])
ax.set_xlabel("Queda do F1-macro ao embaralhar o atributo")
ax.set_title(f"Importância por permutação: {MELHOR}")
fig.tight_layout()
fig.savefig(FIG_DIR / "n2_importancia.png", dpi=200)
plt.show()

## 11. Checagem temporal (treino até 2022, teste 2023 a 2026)

Como na N1, mas com os modelos ajustados e o conjunto ampliado. Agora 2024 também entra no teste. Atenção: `ano_arquivo` é um atributo do conjunto ampliado, e em anos de teste ele fica fora da faixa vista no treino; as árvores tratam esses valores como "o último ano conhecido", então a checagem mede também o custo dessa extrapolação.

In [ ]:
temporal = pd.DataFrame()
if RODAR_TEMPORAL:
    ano = df["ano_arquivo"].to_numpy()
    pos_tr_t = np.flatnonzero(ano <= 2022)
    pos_te_t = np.flatnonzero(ano >= 2023)
    if FRACAO_TREINO < 1:
        pos_tr_t = pos_tr_t[av.amostra_estratificada(y[pos_tr_t], int(FRACAO_TREINO * len(pos_tr_t)), seed=SEED + 3)]
    w_t = compute_sample_weight("balanced", y[pos_tr_t])
    linhas = []
    for conjunto in ["base", "ampliado"]:
        cols = CONJUNTOS[conjunto]
        X_tr_t = matriz(PREP[conjunto], pos_tr_t, cols)
        X_te_t = matriz(PREP[conjunto], pos_te_t, cols)
        for nome, construir in CONSTRUTORES.items():
            pred_t = construir(**MELHORES[nome]).fit(X_tr_t, y[pos_tr_t], sample_weight=w_t).predict(X_te_t)
            for a in sorted(np.unique(ano[pos_te_t])):
                m = ano[pos_te_t] == a
                linhas.append({"modelo": f"{nome} ({conjunto})", "ano_teste": int(a), "n": int(m.sum()),
                               "f1_macro": av.f1_macro(y[pos_te_t][m], pred_t[m], k)})
            linhas.append({"modelo": f"{nome} ({conjunto})", "ano_teste": "2023-2026", "n": len(pos_te_t),
                           "f1_macro": av.f1_macro(y[pos_te_t], pred_t, k)})
        del X_tr_t, X_te_t
    temporal = pd.DataFrame(linhas)
    print(temporal.pivot(index="modelo", columns="ano_teste", values="f1_macro").round(4).to_string())
    temporal.to_csv(RESULTADOS_DIR / "n2_temporal.csv", index=False)
else:
    print("Checagem temporal desligada (RODAR_TEMPORAL = 0).")

## 12. Resumo exportado

Todos os números do artigo final saem dos arquivos em `resultados/` gerados por este notebook. O arquivo `n2_resumo.json` junta os principais.

In [ ]:
resumo = {
    "ambiente": {"python": platform.python_version(), "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
                 "pandas": pd.__version__, "cpus": os.cpu_count(), "ram_gb": ram_gb},
    "config": {"FRACAO_TREINO": FRACAO_TREINO, "N_BUSCA_TREINO": N_BUSCA_TREINO, "N_BUSCA_VAL": N_BUSCA_VAL,
               "N_ITER_BUSCA": N_ITER_BUSCA, "SEMENTES": SEMENTES, "N_BOOT": N_BOOT, "MODO_RAPIDO": MODO_RAPIDO},
    "n": {"total": len(df), "treino": len(idx_treino), "teste": len(idx_teste), "treino_final": len(idx_final),
          "busca_treino": len(idx_busca_tr), "busca_val": len(idx_busca_val)},
    "melhores_parametros": MELHORES,
    "melhor_modelo": MELHOR,
    "comparacao": comparacao.round(6).reset_index().to_dict(orient="records"),
    "diferencas": resumo_dif.round(6).reset_index(names="par").to_dict(orient="records"),
    "sementes": resumo_sementes.round(6).reset_index().to_dict(orient="records"),
    "ablacao": ablacao.round(6).to_dict(orient="records"),
    "importancia": importancia.round(6).to_dict(orient="records"),
    "temporal": temporal.round(6).to_dict(orient="records"),
    "n1_referencia": N1,
}
(RESULTADOS_DIR / "n2_resumo.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"Salvo em {RESULTADOS_DIR / 'n2_resumo.json'}")
print(sorted(p.name for p in RESULTADOS_DIR.glob("n2_*")))

## 13. Discussão dos resultados finais

*Esta seção é escrita depois da execução, a partir das saídas acima e dos arquivos em `resultados/`, e acompanha a seção de resultados do artigo final (`docs/ArtigoFinal-Etapa3.tex`).*